# 0.0.0 WhisperX/Pyannote Transcription+Diarization Pipeline 

This Jupyter notebook is designed to test and evaluate a new Transcription and Diarization Pipeline with the following objectives:
1. Achieving word-level transcription accuracy to ensure detailed and precise text representation of the audio input.
2. Assessing diarization confidence levels to accurately attribute spoken segments to different speakers and measure the reliability of speaker identification.
3. Enhancing the alignment of transcriptions to be closer to natural sentence segments, thereby improving the readability and usability of the transcribed data.

The notebook leverages advanced transcription and diarization capabilities provided by the Whisper, WhisperX, and pyannote libraries. By using GPU acceleration, it processes audio data efficiently, performing alignment and diarization to produce structured outputs that are saved in CSV format for further analysis. The resources and installation instructions are included to facilitate the setup and execution of the pipeline.

Resources:
https://towardsdatascience.com/unlock-the-power-of-audio-data-advanced-transcription-and-diarization-with-whisper-whisperx-and-ed9424307281 

# 0.1 Setup
WhisperX documentation found here: https://github.com/m-bain/whisperX
================================================
1. Install Git
2. Install FFMPEG and add to PATH
3. Install Anaconda 

================================================   
4. Create Conda environment
```sh
conda create -n whisperxtranscription-env python=3.10
conda activate whisperxtranscription-env
```
5. Install PyTorch https://pytorch.org/get-started/locally/ 
```sh
pip install numpy==1.26.3 torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
```
6. Install WhisperX repository and additional packages
```sh
pip install whisperx==3.2.0

pip install speechbrain ipykernel ipywidgets charset-normalizer pandas nltk plotly matplotlib webvtt-py pypi-json srt python-dotenv tqdm

```

7. Create .env file at the same level as this notebook file with the following line
```sh
HF_TOKEN="REPLACEWITHHUGGINGFACETOKENHERE"
```
=================================================
8. For GPU usage :
Install Visual Studio Community https://visualstudio.microsoft.com/downloads/
Install NVIDIA CUDA Toolkit 12.1 https://developer.nvidia.com/cuda-12-1-0-download-archive 

Check PyTorch and CUDA installation
```sh
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
```

=================================================
Fix Numpy
```sh
pip uninstall numpy -y
pip install numpy==1.26.3
```

Fix PyTorch
```sh
pip uninstall torch torchvision torchaudio -y
```

In [2]:
import torch
x = torch.rand(5, 3)
print(x)

tensor([[0.4655, 0.2982, 0.1432],
        [0.0161, 0.6398, 0.9651],
        [0.7475, 0.4195, 0.5348],
        [0.2819, 0.6834, 0.8742],
        [0.7341, 0.7244, 0.6665]])


# 0.2 Check once to see if CUDA GPU is available and PyTorch is working properly

In [1]:
# Check if CUDA GPU is available to PyTorch
import torch                                                # PyTorch
#torch.cuda.set_device(0)                                    # Set the main GPU as device to use if present
print(torch.__version__)
torch.cuda.is_available(),torch.cuda.get_device_name()      # Check if GPU is available and get the name of the GPU

2.4.1


AssertionError: Torch not compiled with CUDA enabled

In [3]:
%pip show whisperx

Name: whisperx
Version: 3.2.0
Summary: Time-Accurate Automatic Speech Recognition using Whisper.
Home-page: https://github.com/m-bain/whisperx
Author: Max Bain
Author-email: 
License: MIT
Location: /opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages
Requires: ctranslate2, faster-whisper, nltk, pandas, pyannote.audio, setuptools, torch, torchaudio, transformers
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy
print(numpy.__version__)

1.26.3


# 1.0 Setup - Start here by adjusting variables
1. choose batch size, compute type, whisper model, and file extension to transcribe

In [5]:
# --- Mac-friendly WhisperX config (Apple Silicon / MPS ready) ---

import os, platform, warnings, logging, gc, datetime, json
from tkinter import Tk, filedialog
import pandas as pd
import torch
import whisperx
import webvtt
import dotenv

# -------------------- Warnings cleanup --------------------
warnings.filterwarnings("ignore", message=".*set_audio_backend has been deprecated.*")
warnings.filterwarnings("ignore", message=".*get_audio_backend has been deprecated.*")
warnings.filterwarnings("ignore", message=".*Module 'speechbrain.pretrained' was deprecated.*")
warnings.filterwarnings("ignore", message=".*AudioMetaData.*moved to.*")
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("speechbrain.utils.quirks").setLevel(logging.WARNING)

# -------------------- Env / tokens --------------------
dotenv.load_dotenv()                    # load .env if present
hf_token = os.getenv("HF_TOKEN", None)  # Hugging Face token (optional)

# -------------------- Device selection --------------------
def pick_device() -> str:
    # Prefer Apple Metal on Apple Silicon
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        # allow per-op CPU fallback for missing MPS kernels
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
        return "mps"
    # Otherwise use CUDA if available
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

device = pick_device()

# -------------------- Compute type & batching --------------------
# Reasonable defaults per device to avoid OOMs
if device == "mps":
    compute_type = "float16"     # MPS runs well with fp16
    batch_size   = 1             # keep small on MPS to avoid spikes
elif device == "cuda":
    compute_type = "float16"     # fp16 on NVIDIA
    batch_size   = 8             # adjust to taste/VRAM
else:
    compute_type = "int8_float16"  # CPU quantized path (faster-whisper)
    batch_size   = 1

# Language / task
language = "en"                  # e.g., "en", "es", "auto" also works in whisperx
task = "transcribe"              # or "translate"

# Model
whisperx_model = "large-v3"      # try "medium" if you hit memory limits

# File types (include common video + audio)
extensions = [
    ".wav", ".mp3", ".flac", ".m4a", ".aac", ".ogg", ".wma",
    ".mp4", ".m4v", ".mov", ".mkv", ".webm"
]

# -------------------- Optional: ASR/VAD tuning --------------------
# Smaller chunks lower peak memory; beam_size=1 reduces decode cost
asr_options = {
    "chunk_length_s": 20,   # try 10–20 if memory is tight
    "beam_size": 1,
    "best_of": 1,
    "patience": 0
}

# -------------------- Load model (example) --------------------
# NOTE: If you’re using faster-whisper via whisperx (recommended), the
# compute_type/device above will be respected.
# model = whisperx.load_model(
#     whisperx_model,
#     device=device,
#     compute_type=compute_type,
#     asr_options=asr_options,
#     language=language,
#     hf_token=hf_token,  # optional
# )

print(f"Device: {device} | compute_type: {compute_type} | batch_size: {batch_size}")


/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/pyannote/audio/core/io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/pyannote/audio/pipelines/speaker_verification.py:43: UserWarning: torchaudio._backend.get_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  backend = torchaudio.get_audio_backend()
/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/pyannote/audio/pipelines/speaker_verification.py:45: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import (
/opt

Device: mps | compute_type: float16 | batch_size: 1


/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/pyannote/audio/tasks/segmentation/mixins.py:37: UserWarning: `torchaudio.backend.common.AudioMetaData` has been moved to `torchaudio.AudioMetaData`. Please update the import path.
  from torchaudio.backend.common import AudioMetaData


# 2.0 Run - after adjusting variables first

Just push run here. You shouldn't need to change anything here unless you want to output less or more file types. These are mostly functions which are then called at the end of the cell.

1. You should get a popup asking to choose the folder where the files are found (It will also search subfolders).

2. You should then get a popup asking for where the transcription files should be placed (It will replicate the folder structure in which they were found)

3. You will also see a popup asking if you want to anonymize with a pseudonyms.csv file, and if so where it is located.

4. You should then see an output similar to the following (just ignore the warnings):

Model was trained with pyannote.audio 0.0.1, yours is 3.1.1. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.3.0+cu121. Bad things might happen unless you revert torch to 1.x.

5. When complete you will see where each were written and the folders where they were written to.


In [6]:
from tkinter import Tk, filedialog, messagebox

# -------------------- Helpers --------------------
def find_audio_files(base_dir, extensions):
    audio_files = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if any(file.lower().endswith(ext.lower()) for ext in extensions):
                audio_files.append(os.path.join(root, file))
    return audio_files

def anonymize_text(text, pseudonym_dict):
    if not pseudonym_dict:
        return text
    # simple exact-match replace; expand to regex if you need boundaries / case-insensitive
    for real_name, pseudonym in pseudonym_dict.items():
        text = text.replace(real_name, pseudonym)
    return text

def format_vtt_timestamp(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{int(hours):02}:{int(minutes):02}:{int(seconds):02}.{milliseconds:03}"

def _empty_device_cache():
    """Free VRAM on CUDA / MPS; safe no-op on CPU."""
    try:
        if device == "cuda" and torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif device == "mps" and hasattr(torch, "mps") and torch.backends.mps.is_available():
            # there is no explicit mps empty_cache API; force a GC helps
            pass
    except Exception:
        pass
    gc.collect()

def save_transcripts(segments, output_dir, relative_path, pseudonym_dict=None):
    # anonymize + sentence numbers
    if pseudonym_dict:
        for seg in segments:
            seg["text"] = anonymize_text(seg["text"], pseudonym_dict)
    for i, seg in enumerate(segments, start=1):
        seg["sentence_number"] = i

    df = pd.DataFrame(segments)
    if "text" in df:
        df["text"] = df["text"].apply(lambda x: x.lstrip())

    cols = ["sentence_number"] + [c for c in df.columns if c != "sentence_number"]
    df = df[cols]

    out_dir = os.path.join(output_dir, os.path.dirname(relative_path))
    os.makedirs(out_dir, exist_ok=True)

    base_filename = os.path.splitext(os.path.basename(relative_path))[0]

    # CSV
    csv_path = os.path.join(out_dir, f"{base_filename}_transcription.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    # TXT
    txt_path = os.path.join(out_dir, f"{base_filename}_transcription.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        for seg in segments:
            f.write(f"{seg['text'].strip()}\n")

    # JSON
    json_path = os.path.join(out_dir, f"{base_filename}_transcription.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(segments, f, ensure_ascii=False, indent=4)

    # VTT
    vtt = webvtt.WebVTT()
    for seg in segments:
        cap = webvtt.Caption()
        cap.start = format_vtt_timestamp(seg["start"])
        cap.end = format_vtt_timestamp(seg["end"])
        cap.lines = [f"{seg['sentence_number']}: {seg['text'].strip()}"]
        vtt.captions.append(cap)
    vtt.save(os.path.join(out_dir, f"{base_filename}_transcription.vtt"))

# -------------------- Core processing --------------------
def process_audio_file(audio_file, base_output_dir, relative_path, pseudonym_dict=None):
    try:
        print(f"Processing {audio_file}...")
        audio = whisperx.load_audio(audio_file)

        # Load ASR (faster-whisper backend) with Mac-safe options
        model = whisperx.load_model(
            whisperx_model,
            device=device,
            compute_type=compute_type,
            asr_options=asr_options,
            language=language,        # can be "auto" too
            #hf_token=hf_token,        # optional
        )

        # transcribe (keep num_workers low on mac)
        result = model.transcribe(
            audio,
            batch_size=batch_size,
            language=language,
            task=task,
            num_workers=0,
        )

        # free decoder memory before alignment
        del model
        _empty_device_cache()

        # Alignment (phoneme-level timing refinement)
        model_a, metadata = whisperx.load_align_model(
            language_code=language if language != "auto" else "en",
            device=device,
        )
        result = whisperx.align(
            result["segments"],
            model_a,
            metadata,
            audio,
            device,
            return_char_alignments=False,
        )
        del model_a
        _empty_device_cache()

        # Diarization
        diarize_model = whisperx.DiarizationPipeline(
            device=device,
            use_auth_token=hf_token
        )
        diarize_segments = diarize_model(audio)
        result = whisperx.assign_word_speakers(diarize_segments, result)

        # Save
        save_transcripts(result["segments"], base_output_dir, relative_path, pseudonym_dict)

    except RuntimeError as re:
        # Adaptive retry on OOM / MPS fallback hiccups
        if "out of memory" in str(re).lower():
            print("OOM hit — retrying with smaller chunk_length and batch size...")
            try:
                smaller_opts = {**asr_options, "chunk_length_s": max(10, int(asr_options.get("chunk_length_s", 20) // 2))}
                model = whisperx.load_model(
                    whisperx_model,
                    device=device,
                    compute_type=compute_type,
                    asr_options=smaller_opts,
                    language=language,
                    #hf_token=hf_token,
                )
                result = model.transcribe(
                    audio,
                    batch_size=1,
                    language=language,
                    task=task,
                    num_workers=0,
                )
                del model
                _empty_device_cache()

                model_a, metadata = whisperx.load_align_model(
                    language_code=language if language != "auto" else "en",
                    device=device,
                )
                result = whisperx.align(
                    result["segments"], model_a, metadata, audio, device, return_char_alignments=False
                )
                del model_a
                _empty_device_cache()

                diarize_model = whisperx.DiarizationPipeline(device=device, use_auth_token=hf_token)
                diarize_segments = diarize_model(audio)
                result = whisperx.assign_word_speakers(diarize_segments, result)

                save_transcripts(result["segments"], base_output_dir, relative_path, pseudonym_dict)
            except Exception as e2:
                import traceback
                print(f"Retry also failed for {audio_file}:\n{traceback.format_exc()}")
        else:
            import traceback
            print(f"Error processing {audio_file}:\n{traceback.format_exc()}")

    except Exception:
        import traceback
        print(f"Error processing {audio_file}:\n{traceback.format_exc()}")

def main():
    # Initialize Tkinter (front-most prompts)
    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    # Choose input/output
    input_folder = filedialog.askdirectory(title="Select Folder Containing Audio/Video Files")
    if not input_folder:
        print("No folder selected. Exiting.")
        return

    output_folder = filedialog.askdirectory(title="Select Folder to Save Transcriptions")
    if not output_folder:
        print("No output folder selected. Exiting.")
        return

    # Optional pseudonyms
    use_pseudonyms = messagebox.askyesno(
        "Pseudonyms",
        "Use a pseudonyms.csv file to anonymize the transcripts?"
    )
    pseudonym_dict = None
    if use_pseudonyms:
        pseudonyms_file = filedialog.askopenfilename(
            title="Select Pseudonyms CSV File",
            filetypes=[("CSV files", "*.csv")]
        )
        if pseudonyms_file:
            pseudonyms_df = pd.read_csv(pseudonyms_file)
            # Expecting columns: name,pseudonym
            pseudonym_dict = dict(zip(pseudonyms_df["name"], pseudonyms_df["pseudonym"]))
            print(f"Pseudonyms loaded from {pseudonyms_file}.")
        else:
            print("No pseudonyms file selected. Continuing without pseudonymization.")

    # Discover files
    audio_files = find_audio_files(input_folder, extensions)
    print(f"Found {len(audio_files)} files to process.")

    # Process
    for audio_file in audio_files:
        relative_path = os.path.relpath(audio_file, input_folder)
        process_audio_file(audio_file, output_folder, relative_path, pseudonym_dict)
        print(f"Processed {audio_file}")

    print("All files processed.")

if __name__ == "__main__":
    main()


2025-11-10 08:48:26.794 python[45369:14192132] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


Found 2 files to process.
Processing /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue1.ogg...
Error processing /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue1.ogg:
Traceback (most recent call last):
  File "/var/folders/19/6pjx738x259c707gbz3_6cd80000gn/T/ipykernel_45369/2365795919.py", line 90, in process_audio_file
    model = whisperx.load_model(
  File "/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/whisperx/asr.py", line 288, in load_model
    model = model or WhisperModel(whisper_arch,
  File "/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/faster_whisper/transcribe.py", line 145, in __init__
    self.model = ctranslate2.models.Whisper(
ValueError: unsupported device mps

Processed /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue1.ogg
Processing /Users/kevinhall/Documents/AANLP_Work/WhisperXTra

In [2]:
from tkinter import Tk, filedialog, messagebox

# Functions
def find_audio_files(base_dir, extensions):
    audio_files = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if any(file.lower().endswith(ext.lower()) for ext in extensions):
                audio_files.append(os.path.join(root, file))
    return audio_files

def anonymize_text(text, pseudonym_dict):
    for real_name, pseudonym in pseudonym_dict.items():
        text = text.replace(real_name, pseudonym)
    return text

def format_vtt_timestamp(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{int(hours):02}:{int(minutes):02}:{int(seconds):02}.{milliseconds:03}"

def save_transcripts(segments, output_dir, relative_path, pseudonym_dict=None):
    if pseudonym_dict:
        for segment in segments:
            segment['text'] = anonymize_text(segment['text'], pseudonym_dict)
    for i, segment in enumerate(segments):
        segment['sentence_number'] = i + 1
    df = pd.DataFrame(segments)
    df['text'] = df['text'].apply(lambda x: x.lstrip())
    cols = ['sentence_number'] + [col for col in df.columns if col != 'sentence_number']
    df = df[cols]

    os.makedirs(output_dir, exist_ok=True)
    base_filename = os.path.splitext(os.path.basename(relative_path))[0]
    csv_path = os.path.join(output_dir, f"{base_filename}_transcription.csv")
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')

    with open(os.path.join(output_dir, f"{base_filename}_transcription.txt"), 'w', encoding='utf-8') as f:
        for segment in segments:
            f.write(f"{segment['text'].strip()}\n")

    json_path = os.path.join(output_dir, f"{base_filename}_transcription.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(segments, f, ensure_ascii=False, indent=4)

    vtt = webvtt.WebVTT()
    for segment in segments:
        caption = webvtt.Caption()
        caption.start = format_vtt_timestamp(segment['start'])
        caption.end = format_vtt_timestamp(segment['end'])
        caption.lines = [f"{segment['sentence_number']}: {segment['text'].strip()}"]
        vtt.captions.append(caption)
    vtt.save(os.path.join(output_dir, f"{base_filename}_transcription.vtt"))

def process_audio_file(audio_file, base_output_dir, relative_path, pseudonym_dict=None):
    try:
        print(f"Processing {audio_file}...")
        audio = whisperx.load_audio(audio_file)
        model = whisperx.load_model(whisperx_model, device, compute_type=compute_type)
        result = model.transcribe(audio, batch_size=batch_size, language=language, task=task)
        del model; gc.collect(); torch.cuda.empty_cache()

        model_a, metadata = whisperx.load_align_model(language_code=language, device=device)
        result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
        del model_a; gc.collect(); torch.cuda.empty_cache()

        # Correct way to load diarization model in recent whisperx
        diarize_model = whisperx.DiarizationPipeline(use_auth_token=hf_token, device=device)
        diarize_segments = diarize_model(audio)
        result = whisperx.assign_word_speakers(diarize_segments, result)

        output_dir = os.path.join(base_output_dir, os.path.dirname(relative_path))
        save_transcripts(result["segments"], output_dir, relative_path, pseudonym_dict)
    except Exception as e:
        import traceback
        print(f"Error processing {audio_file}:\n{traceback.format_exc()}")

def main():
    # Initialize Tkinter
    root = Tk()
    root.withdraw()  # Hide the main window
    
    # Bring the root window to the front
    root.attributes('-topmost', True)

    # Popup for input folder
    input_folder = filedialog.askdirectory(title="Select Folder Containing Audio/Video Files")
    if not input_folder:
        print("No folder selected. Exiting.")
        return

    # Popup for output folder
    output_folder = filedialog.askdirectory(title="Select Folder to Save Transcriptions")
    if not output_folder:
        print("No output folder selected. Exiting.")
        return

    # Ask if a pseudonyms.csv file will be used
    use_pseudonyms = messagebox.askyesno("Pseudonyms", "Will you use a pseudonyms.csv file for to anonymize the transcripts?")
    pseudonym_dict = None

    if use_pseudonyms:
        pseudonyms_file = filedialog.askopenfilename(
            title="Select Pseudonyms CSV File",
            filetypes=[("CSV files", "*.csv")]
        )
        if not pseudonyms_file:
            print("No pseudonyms file selected. Continuing without pseudonymization.")
        else:
            # Load the pseudonyms file
            pseudonyms_df = pd.read_csv(pseudonyms_file)
            pseudonym_dict = dict(zip(pseudonyms_df['name'], pseudonyms_df['pseudonym']))
            print(f"Pseudonyms loaded from {pseudonyms_file}.")

    # Find and process audio files
    audio_files = find_audio_files(input_folder, extensions)
    print(f"Found {len(audio_files)} files to process.")

    for audio_file in audio_files:
        relative_path = os.path.relpath(audio_file, input_folder)
        process_audio_file(audio_file, output_folder, relative_path, pseudonym_dict)
        print(f"Processed {audio_file}")

    print("All files processed.")

if __name__ == "__main__":
    main()


2025-11-05 15:38:34.455 python[94773:11521622] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


Found 2 files to process.
Processing /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue1.ogg...


vocabulary.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Error processing /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue1.ogg:
Traceback (most recent call last):
  File "/var/folders/19/6pjx738x259c707gbz3_6cd80000gn/T/ipykernel_94773/1870150986.py", line 60, in process_audio_file
    model = whisperx.load_model(whisperx_model, device, compute_type=compute_type)
  File "/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/whisperx/asr.py", line 288, in load_model
    model = model or WhisperModel(whisper_arch,
  File "/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/faster_whisper/transcribe.py", line 133, in __init__
    self.model = ctranslate2.models.Whisper(
ValueError: unsupported device mps

Processed /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue1.ogg
Processing /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue2.ogg...
Error processing /Users/kevinhall/Do

In [1]:
# --- WhisperX batch transcriber: Apple Silicon (MPS) friendly ---
# - ASR (Faster-Whisper/ctranslate2) runs on CPU (int8_float16 by default)
# - Alignment & diarization use MPS if available (else CPU)
# - Handles audio + common video containers, pseudonymization, and CSV/TXT/JSON/VTT outputs

import os, platform, warnings, logging, gc, datetime, json
from tkinter import Tk, filedialog, messagebox
import pandas as pd
import torch
import whisperx
import webvtt
import dotenv

# -------------------- Warnings cleanup --------------------
warnings.filterwarnings("ignore", message=".*set_audio_backend has been deprecated.*")
warnings.filterwarnings("ignore", message=".*get_audio_backend has been deprecated.*")
warnings.filterwarnings("ignore", message=".*Module 'speechbrain.pretrained' was deprecated.*")
warnings.filterwarnings("ignore", message=".*AudioMetaData.*moved to.*")
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("speechbrain.utils.quirks").setLevel(logging.WARNING)

# -------------------- Env / tokens --------------------
dotenv.load_dotenv()                              # load .env if present
hf_token = os.getenv("HF_TOKEN", None)            # used by diarization model if private

# Optional: give ctranslate2 (CPU) more threads (tune to your cores)
os.environ.setdefault("OMP_NUM_THREADS", "6")
os.environ.setdefault("CT2_NUM_THREADS", "6")

# -------------------- Device selection --------------------
def detect_devices():
    has_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    has_cuda = torch.cuda.is_available()

    # Torch ops (alignment/diarization)
    torch_device = "cuda" if has_cuda else ("mps" if has_mps else "cpu")
    if torch_device == "mps":
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

    # ASR (ctranslate2) supports only "cuda" or "cpu" — not "mps"
    asr_device = "cuda" if has_cuda else "cpu"

    # Sensible defaults
    if asr_device == "cpu":
        asr_compute = "int8"   # fast & accurate on Apple CPUs
        asr_batch   = 1
    else:  # CUDA
        asr_compute = "float16"
        asr_batch   = 8

    return {
        "asr_device": asr_device,
        "asr_compute": asr_compute,
        "asr_batch": asr_batch,
        "torch_device": torch_device,
    }

DEV = detect_devices()
asr_device   = DEV["asr_device"]     # "cpu" on Macs without NVIDIA
compute_type = DEV["asr_compute"]    # e.g., "int8_float16"
batch_size   = DEV["asr_batch"]      # 1 on CPU
torch_device = DEV["torch_device"]   # "mps" if available

# -------------------- Task/Model/Options --------------------
language       = "en"          # or "auto"
task           = "transcribe"  # or "translate"
whisperx_model = "large-v3"    # use "medium" if you want less RAM/VRAM
extensions = [
    ".wav", ".mp3", ".flac", ".m4a", ".aac", ".ogg", ".wma",
    ".mp4", ".m4v", ".mov", ".mkv", ".webm"
]
asr_options = {
    "chunk_length_s": 20,  # drop to 10 if you hit memory spikes
    "beam_size": 1,
    "best_of": 1,
    "patience": 0,
}

# -------------------- Helpers --------------------
def _empty_device_cache():
    """Free memory pressure. CUDA has explicit empty; MPS/CPU rely on GC."""
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    gc.collect()

def find_audio_files(base_dir, extensions):
    audio_files = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if any(file.lower().endswith(ext.lower()) for ext in extensions):
                audio_files.append(os.path.join(root, file))
    return audio_files

def anonymize_text(text, pseudonym_dict):
    if not pseudonym_dict:
        return text
    for real_name, pseudonym in pseudonym_dict.items():
        text = text.replace(real_name, pseudonym)  # simple exact-match; swap to regex for smarter matching
    return text

def format_vtt_timestamp(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{int(hours):02}:{int(minutes):02}:{int(seconds):02}.{milliseconds:03}"

def save_transcripts(segments, output_dir, relative_path, pseudonym_dict=None):
    # anonymize + sentence numbers
    if pseudonym_dict:
        for seg in segments:
            seg["text"] = anonymize_text(seg["text"], pseudonym_dict)
    for i, seg in enumerate(segments, start=1):
        seg["sentence_number"] = i

    df = pd.DataFrame(segments)
    if "text" in df:
        df["text"] = df["text"].apply(lambda x: x.lstrip())
    cols = ["sentence_number"] + [c for c in df.columns if c != "sentence_number"]
    df = df[cols]

    out_dir = os.path.join(output_dir, os.path.dirname(relative_path))
    os.makedirs(out_dir, exist_ok=True)
    base_filename = os.path.splitext(os.path.basename(relative_path))[0]

    # CSV
    csv_path = os.path.join(out_dir, f"{base_filename}_transcription.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    # TXT
    with open(os.path.join(out_dir, f"{base_filename}_transcription.txt"), "w", encoding="utf-8") as f:
        for seg in segments:
            f.write(f"{seg['text'].strip()}\n")

    # JSON
    with open(os.path.join(out_dir, f"{base_filename}_transcription.json"), "w", encoding="utf-8") as f:
        json.dump(segments, f, ensure_ascii=False, indent=4)

    # VTT
    vtt = webvtt.WebVTT()
    for seg in segments:
        cap = webvtt.Caption()
        cap.start = format_vtt_timestamp(seg["start"])
        cap.end = format_vtt_timestamp(seg["end"])
        cap.lines = [f"{seg['sentence_number']}: {seg['text'].strip()}"]
        vtt.captions.append(cap)
    vtt.save(os.path.join(out_dir, f"{base_filename}_transcription.vtt"))

# -------------------- Core processing --------------------
def process_audio_file(audio_file, base_output_dir, relative_path, pseudonym_dict=None):
    try:
        print(f"Processing {audio_file}...")
        audio = whisperx.load_audio(audio_file)

        # --- ASR (ctranslate2 backend) on CPU (or CUDA if you have it) ---
        model = whisperx.load_model(
            whisperx_model,
            device=asr_device,            # "cpu" on Mac
            compute_type=compute_type,    # e.g., "int8_float16"
            asr_options=asr_options,
            language=language,
        )
        result = model.transcribe(
            audio,
            batch_size=batch_size,        # 1 on CPU
            language=language,
            task=task,
            num_workers=0,                # mac-friendly
        )
        del model
        _empty_device_cache()

        # --- Alignment on MPS (or CPU) ---
        model_a, metadata = whisperx.load_align_model(
            language_code=language if language != "auto" else "en",
            device=torch_device,
        )
        result = whisperx.align(
            result["segments"],
            model_a,
            metadata,
            audio,
            torch_device,
            return_char_alignments=False,
        )
        del model_a
        _empty_device_cache()

        # --- Diarization on MPS (or CPU) ---
        diarize_model = whisperx.DiarizationPipeline(
            device=torch_device,
            use_auth_token=hf_token
        )
        diarize_segments = diarize_model(audio)
        result = whisperx.assign_word_speakers(diarize_segments, result)

        # --- Save ---
        save_transcripts(result["segments"], base_output_dir, relative_path, pseudonym_dict)

    except RuntimeError as re:
        if "out of memory" in str(re).lower():
            print("OOM hit — retrying with smaller chunk_length and batch size...")
            try:
                smaller_opts = {**asr_options, "chunk_length_s": max(10, int(asr_options.get("chunk_length_s", 20) // 2))}
                model = whisperx.load_model(
                    whisperx_model,
                    device=asr_device,          # keep on CPU
                    compute_type=compute_type,
                    asr_options=smaller_opts,
                    language=language,
                )
                result = model.transcribe(
                    audio,
                    batch_size=1,
                    language=language,
                    task=task,
                    num_workers=0,
                )
                del model
                _empty_device_cache()

                model_a, metadata = whisperx.load_align_model(
                    language_code=language if language != "auto" else "en",
                    device=torch_device,
                )
                result = whisperx.align(result["segments"], model_a, metadata, audio, torch_device, return_char_alignments=False)
                del model_a
                _empty_device_cache()

                diarize_model = whisperx.DiarizationPipeline(device=torch_device, use_auth_token=hf_token)
                diarize_segments = diarize_model(audio)
                result = whisperx.assign_word_speakers(diarize_segments, result)

                save_transcripts(result["segments"], base_output_dir, relative_path, pseudonym_dict)
            except Exception:
                import traceback
                print(f"Retry also failed for {audio_file}:\n{traceback.format_exc()}")
        else:
            import traceback
            print(f"Error processing {audio_file}:\n{traceback.format_exc()}")
    except Exception:
        import traceback
        print(f"Error processing {audio_file}:\n{traceback.format_exc()}")

# -------------------- UI / main --------------------
def main():
    # Initialize Tkinter (front-most prompts)
    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    # Choose input/output
    input_folder = filedialog.askdirectory(title="Select Folder Containing Audio/Video Files")
    if not input_folder:
        print("No folder selected. Exiting.")
        return

    output_folder = filedialog.askdirectory(title="Select Folder to Save Transcriptions")
    if not output_folder:
        print("No output folder selected. Exiting.")
        return

    # Optional pseudonyms
    use_pseudonyms = messagebox.askyesno(
        "Pseudonyms",
        "Use a pseudonyms.csv file to anonymize the transcripts?"
    )
    pseudonym_dict = None
    if use_pseudonyms:
        pseudonyms_file = filedialog.askopenfilename(
            title="Select Pseudonyms CSV File",
            filetypes=[("CSV files", "*.csv")]
        )
        if pseudonyms_file:
            pseudonyms_df = pd.read_csv(pseudonyms_file)
            # Expecting columns: name,pseudonym
            pseudonym_dict = dict(zip(pseudonyms_df["name"], pseudonyms_df["pseudonym"]))
            print(f"Pseudonyms loaded from {pseudonyms_file}.")
        else:
            print("No pseudonyms file selected. Continuing without pseudonymization.")

    # Discover files
    audio_files = find_audio_files(input_folder, extensions)
    print(f"Found {len(audio_files)} files to process.")

    # Process
    for audio_file in audio_files:
        relative_path = os.path.relpath(audio_file, input_folder)
        process_audio_file(audio_file, output_folder, relative_path, pseudonym_dict)
        print(f"Processed {audio_file}")

    print("All files processed.")

if __name__ == "__main__":
    print(f"ASR device: {asr_device} | Torch device: {torch_device} | compute_type: {compute_type} | batch_size: {batch_size}")
    main()


/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/pyannote/audio/core/io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/pyannote/audio/pipelines/speaker_verification.py:43: UserWarning: torchaudio._backend.get_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  backend = torchaudio.get_audio_backend()
/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/pyannote/audio/pipelines/speaker_verification.py:45: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import (
/opt

ASR device: cpu | Torch device: mps | compute_type: int8 | batch_size: 1


2025-11-05 16:01:26.389 python[97450:11550002] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


Found 2 files to process.
Processing /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue1.ogg...
Error processing /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue1.ogg:
Traceback (most recent call last):
  File "/var/folders/19/6pjx738x259c707gbz3_6cd80000gn/T/ipykernel_97450/4089955278.py", line 158, in process_audio_file
    model = whisperx.load_model(
  File "/opt/miniconda3/envs/whisperx-mps/lib/python3.10/site-packages/whisperx/asr.py", line 334, in load_model
    default_asr_options = faster_whisper.transcribe.TranscriptionOptions(**default_asr_options)
TypeError: TranscriptionOptions.__new__() got an unexpected keyword argument 'chunk_length_s'

Processed /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue1.ogg
Processing /Users/kevinhall/Documents/AANLP_Work/WhisperXTranscription4Researchers/Data/rawAudioFiles/Monologue2.ogg...
E

In [2]:
import faster_whisper, av
from faster_whisper.transcribe import TranscriptionOptions
print("faster-whisper:", faster_whisper.__version__)
print("pyav:", av.__version__)
print("TranscriptionOptions params:", TranscriptionOptions.__init__.__code__.co_varnames)


faster-whisper: 1.0.3
pyav: 11.0.0


AttributeError: 'wrapper_descriptor' object has no attribute '__code__'